# Fine-tuning Moirai — Adapter Supervisado

**Pipeline (sin leakage):**

| Partición | Uso |
|-----------|-----|
| `df_70` → train (90%) | Cachear preds zero-shot + entrenar adaptador |
| `df_70` → val (10%) | Monitoreo loss durante entrenamiento |
| `df_30` (30%) | Evaluación final — nunca visto durante entrenamiento |

**Estrategia:** Moirai base congelado → predicciones zero-shot cacheadas → adaptador residual multicapa (cross-attention + gate aprendible).  
Al finalizar se guarda `nueva_info/moirai_adapter.pt` para uso en notebooks de ensemble.

In [3]:
import sys
# En Colab instala las dependencias; localmente ya están en el venv — esta celda es segura en ambos casos
!{sys.executable} -m pip install -q uni2ts gluonts einops scikit-learn pandas openpyxl pyyaml matplotlib joblib

/home/allandbb/Documents/GitHub/heartrate-forecasting/.venv/bin/python: No module named pip


In [2]:
import os
import sys
import importlib

# ---- Colab: clonar repo ----
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.chdir('/content')
    if not os.path.exists('heartrate-forecasting'):
        !git clone https://github.com/AllanDBB/heartrate-forecasting.git
    os.chdir('heartrate-forecasting')

if not IN_COLAB:
    try:
        _nb_path = __vsc_ipynb_file__
        REPO_DIR = os.path.dirname(os.path.dirname(os.path.abspath(_nb_path)))
    except NameError:
        REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
else:
    REPO_DIR = os.getcwd()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'REPO_DIR: {REPO_DIR}')
print(f'Colab: {IN_COLAB}')

import main
import utils
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

importlib.reload(utils)
importlib.reload(main)

REPO_DIR: /home/allandbb/Documents/GitHub/heartrate-forecasting
Colab: False


<module 'main' from '/home/allandbb/Documents/GitHub/heartrate-forecasting/wrappers/../wrappers/../main.py'>

## 1. Configuración

In [ ]:
INPUT_SIZE  = 200
OUTPUT_SIZE = 200
CACHE_DIR   = 'cache_moirai'
SEED        = 42
CONFIG_PATH = 'configs/moirai_config.yaml'
SAVE_PATH   = 'nueva_info/moirai_adapter.pt'

main.ensure_dir(CACHE_DIR)
os.makedirs('nueva_info', exist_ok=True)
print(f'Config: {CONFIG_PATH}')
print(f'Guardará adaptador en: {SAVE_PATH}')

## 2. Carga y preparación de datos

In [ ]:
df_70, df_30, split_meta = main.load_split_dataframes(
    dataset_dir='dataset',
    split_seed=SEED,
    split_70_path='nueva_info/df_70.csv',
    split_30_path='nueva_info/df_30.csv',
)
df_70, df_30, overlap = utils.sanitize_split_dataframes(df_70, df_30)
print(f'Overlap eliminado: {overlap}')

# Estandarizar cada split por separado (sin filtración de información)
df_scaled_70, params_70 = utils.estandarizar(df_70, os.path.join(CACHE_DIR, 'values_deses_70.csv'))
df_scaled_30, params_30 = utils.estandarizar(df_30, os.path.join(CACHE_DIR, 'values_deses_30.csv'))

print(f'df_70 estandarizado: {df_scaled_70.shape}')
print(f'df_30 estandarizado: {df_scaled_30.shape}')

In [ ]:
# Ventanas supervisadas del 70% (para fine-tuning)
X_70, y_70, ids_70 = utils.series_to_supervised_matrix(
    df_scaled_70, input_size=INPUT_SIZE, output_size=OUTPUT_SIZE
)
ids_70 = np.array(ids_70)
print(f'Ventanas del 70%: X={X_70.shape}, y={y_70.shape}')

# Train/val split dentro del 70%
X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X_70, y_70, ids_70,
    test_size=0.1,
    random_state=SEED,
    stratify=ids_70,
)
print(f'Train: X={X_train.shape}, Val: X={X_val.shape}')

# Ventanas del 30% para evaluación final
X_30, y_30, ids_30 = utils.series_to_supervised_matrix(
    df_scaled_30, input_size=INPUT_SIZE, output_size=OUTPUT_SIZE
)
ids_30 = np.array(ids_30)
print(f'Ventanas del 30% (held-out): X={X_30.shape}')

## 3. Fine-tuning de Moirai

In [ ]:
import wrappers.MoiraiSupervisedWrapper as _moirai_mod
importlib.reload(_moirai_mod)
from wrappers.MoiraiSupervisedWrapper import MoiraiSupervisedWrapper

wrapper = MoiraiSupervisedWrapper(CONFIG_PATH)
print('Modelo cargado.')

In [ ]:
# Fine-tuning: cachea preds zero-shot una sola vez, luego entrena el adaptador
wrapper.fit(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    epochs=20,
    lr=5e-4,
)

## 4. Evaluación

In [ ]:
# Zero-shot baseline (sin adaptador) sobre held-out
print('Evaluando zero-shot baseline...')
y_base = wrapper._predict_raw(X_30)
y_30_orig  = utils.desestandarizar_ventanas(y_30, ids_30, params_30)
y_base_orig = utils.desestandarizar_ventanas(y_base, ids_30, params_30)

metrics_base = utils.evaluate_all_metrics(y_30_orig, y_base_orig)
print(f"\nZero-shot  MAPE={metrics_base['MAPE']:.4f}  Pearson={metrics_base['Pearson']:.4f}  DTW={metrics_base['DTW']:.4f}")

In [ ]:
import torch as _torch

# Aplica el adaptador sobre las preds zero-shot ya cacheadas — evita correr Moirai dos veces
print('Evaluando modelo fine-tuned (adaptador sobre preds cacheadas)...')
_device = next(wrapper.adapter.parameters()).device
_x_t = _torch.tensor(X_30, dtype=_torch.float32).to(_device)
_b_t = _torch.tensor(y_base, dtype=_torch.float32).to(_device)
with _torch.no_grad():
    y_ft = wrapper.adapter(_x_t, _b_t).cpu().numpy()
y_ft_orig = utils.desestandarizar_ventanas(y_ft, ids_30, params_30)

metrics_ft = utils.evaluate_all_metrics(y_30_orig, y_ft_orig)
print(f"Fine-tuned MAPE={metrics_ft['MAPE']:.4f}  Pearson={metrics_ft['Pearson']:.4f}  DTW={metrics_ft['DTW']:.4f}")

print(f"\nMejora MAPE:    {metrics_base['MAPE'] - metrics_ft['MAPE']:+.4f}")
print(f"Mejora Pearson: {metrics_ft['Pearson'] - metrics_base['Pearson']:+.4f}")

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

results = {
    'Moirai Zero-shot': metrics_base,
    'Moirai Fine-tuned': metrics_ft,
}
utils.plot_metrics_comparison(results, title='Moirai: Zero-shot vs Fine-tuned (held-out 30%)')

In [ ]:
utils.plot_forecast_samples(
    y_30_orig, y_ft_orig, n_samples=6,
    title='Moirai Fine-tuned vs Real (held-out)'
)

In [ ]:
utils.plot_error_over_horizon(
    y_30_orig, y_ft_orig,
    title='Moirai Fine-tuned: Error por horizonte (held-out)'
)

## 5. Guardar modelo

In [ ]:
import torch
import json

# Guardar state dict del adaptador
save_abs = os.path.join(REPO_DIR, SAVE_PATH)
torch.save(wrapper.adapter.state_dict(), save_abs)
print(f'Adaptador guardado en: {save_abs}')
print(f'Tamaño: {os.path.getsize(save_abs) / 1e6:.2f} MB')

# Guardar gate value y métricas como referencia
gate_val = torch.sigmoid(wrapper.adapter.gate).item()
meta = {
    'gate': gate_val,
    'input_size': INPUT_SIZE,
    'output_size': OUTPUT_SIZE,
    'adapter_hidden': 512,
    'metrics_zero_shot': metrics_base,
    'metrics_fine_tuned': metrics_ft,
}
meta_path = save_abs.replace('.pt', '_meta.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Metadatos guardados en: {meta_path}')

print(f'\n=== Resumen final ===')
print(f'Gate aprendido: {gate_val:.3f} (0=solo base Moirai, 1=corrección completa del adaptador)')
print(f'Zero-shot  MAPE={metrics_base["MAPE"]:.4f}  Pearson={metrics_base["Pearson"]:.4f}')
print(f'Fine-tuned MAPE={metrics_ft["MAPE"]:.4f}  Pearson={metrics_ft["Pearson"]:.4f}')

## 6. Verificación de carga (para ensemble)

Confirma que el adaptador se puede recargar y produce las mismas predicciones.

In [ ]:
from wrappers.MoiraiSupervisedWrapper import MoiraiSupervisedWrapper, _ForecastAdapter

# Recrear wrapper con modelo base
wrapper_reload = MoiraiSupervisedWrapper(CONFIG_PATH)

# Cargar adaptador guardado
wrapper_reload.adapter = _ForecastAdapter(INPUT_SIZE, OUTPUT_SIZE, hidden=512)
wrapper_reload.adapter.load_state_dict(torch.load(save_abs, map_location='cpu'))
wrapper_reload.adapter.eval()

# Verificar que las predicciones son idénticas
y_reload = wrapper_reload.predict(X_30[:10])
y_orig   = wrapper.predict(X_30[:10])
max_diff = np.abs(y_reload - y_orig).max()
print(f'Diferencia máxima entre predicciones originales y recargadas: {max_diff:.8f}')
print('OK: el adaptador se carga correctamente.' if max_diff < 1e-4 else 'ADVERTENCIA: diferencia inesperada.')